# 01 · Analysis table: London wards
One row per 2018 ward with the 2016 prescribing outcomes, the nine greenery measures and the four controls (Census 2011). This is the table every model reads.

**Inputs:** `data/prescriptions/ward_prescriptions_2016_*.csv`, `data/greenery_metrics/*_ward.csv`, `data/greenery_metrics/official_indices/wards_official_indices.csv`, `data/raw_data/controls/*`, `data/raw_data/density_data/extracted_per_ward.csv`, ward boundaries.
**Outputs:** `data/data_analysis_wards7s.csv` (controls as described below; greenery and outcomes in their own units, not standardised; the models standardise per sample) and `data/data_analysis_wards7s_geo.geojson`.
**Runtime:** under a minute. Run from this folder.

In [1]:
import geopandas as gpd
import numpy as np
import pandas as pd
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]   # run from this folder
DATA = REPO / 'data'
RAW = DATA / 'raw_data'
GM = DATA / 'greenery_metrics'
LOOKUP = pd.read_csv(RAW / 'lookups/Lower_Layer_Super_Output_Area_(2011)_to_Ward_(2018)_Lookup_in_England_and_Wales_v3.csv')
OUTCOMES = ['diabetes', 'hypertension', 'asthma', 'opioids', 'total']
LEVEL, KEY = 'ward', 'ward_code'

## 1. Prescribing: yearly quantity per registered patient, 2016

In [2]:
data = None
for o in OUTCOMES:
    df = pd.read_csv(DATA / f'prescriptions/ward_prescriptions_2016_{o}.csv')
    df = df[[ 'WD18CD', 'Total_quantity_per_capita']].rename(columns={'WD18CD': KEY, 'Total_quantity_per_capita': f'{o} prescriptions'})
    data = df if data is None else data.merge(df, on=KEY)
print(len(data), 'wards')

637 wards


## 2. Greenery measures
NDVI and our five measures are shares (0–1) of the area, road strip or street-view pixels; the official measures are shares of residents with access.

In [3]:
# file stem -> {source column: name in the analysis table}
SOURCES = {
    'SAT_greenery_on_off_road': {'on_road_green_share': 'SAT_on_road_share_u', 'on_road_green_share_weighted': 'SAT_on_road_share_w'},
    'SAT_greenery_off_road_accessibility': {'accessible_off_road_green': 'SAT_off_road'},
    'GSV_greenery_per': {'GSV_on_road': 'GSV_on_road_u', 'GSV_on_road_weighted': 'GSV_on_road_w'},
}
greenery = pd.read_csv(GM / f'SAT_greenery_total_{LEVEL}.csv')[[KEY, 'percent_green']].rename(columns={'percent_green': 'SAT_NDVI'})
for stem, cols in SOURCES.items():
    df = pd.read_csv(GM / f'{stem}_{LEVEL}.csv')
    greenery = greenery.merge(df[[KEY, *cols]].rename(columns=cols), on=KEY, how='left')
assert greenery[KEY].is_unique
data = data.merge(greenery, on=KEY)
print(len(data), 'areas with prescribing and greenery')
official = pd.read_csv(GM / 'official_indices/wards_official_indices.csv').rename(columns={'GSS_CODE': KEY})
data = data.merge(official[[KEY, 'WHO', 'ESA_WHO_exposure', 'Natural_England1']], on=KEY, how='left')

637 areas with prescribing and greenery


## 3. Controls (Census 2011)
- `median age`: the population-weighted mean of the LSOA medians.
- `white percent`: share of residents who are White.
- `IMD deprivation score`: share of households deprived in 2 of the 4 Census dimensions (a Census measure, not the Index of Multiple Deprivation; the name is kept for the model code).
- `building_density`: mean relative building height (`relh2`) from the GLA building footprints, log(x + 1).

In [4]:
def to_wards(df, sums):
    """Sum LSOA counts to 2018 wards."""
    return df.merge(LOOKUP[['LSOA11CD', 'WD18CD']], left_on='geography code', right_on='LSOA11CD').groupby('WD18CD')[sums].sum()


age = pd.read_csv(RAW / 'controls/age.csv')
age['n'] = age['Age: All usual residents; measures: Value']
age['median_x_n'] = age['Age: Median Age; measures: Value'] * age['n']
age = to_wards(age, ['n', 'median_x_n'])
eth = to_wards(pd.read_csv(RAW / 'controls/ethnic_group.csv'),
               ['Ethnic Group: White; measures: Value', 'Ethnic Group: All usual residents; measures: Value'])
dep = to_wards(pd.read_csv(RAW / 'controls/IMD.csv'),
               ['Household Deprivation: Household is deprived in 2 dimensions; measures: Value',
                'Household Deprivation: All categories: Classification of household deprivation; measures: Value'])
controls = pd.DataFrame({
    'median age': age['median_x_n'] / age['n'],
    'white percent': eth.iloc[:, 0] / eth.iloc[:, 1],
    'IMD deprivation score': dep.iloc[:, 0] / dep.iloc[:, 1],
}).rename_axis(KEY).reset_index()
height = pd.read_csv(RAW / 'density_data/extracted_per_ward.csv')[['ward_code', 'relh2']]
data = data.merge(controls, on=KEY)
data = data.merge(height.rename(columns={'relh2': 'building_density'}), on=KEY, how='left')
missing = data.loc[data['building_density'].isna(), KEY].tolist()
print(f'{len(missing)} wards without building height, dropped: {missing}')
data = data.dropna(subset=['building_density'])
data['building_density'] = np.log(data['building_density'] + 1)

5 wards without building height, dropped: ['E05009288', 'E05009302', 'E05009304', 'E05009308', 'E05009311']


## 4. Centroids (British National Grid metres, for MGWR) → SAVE

In [5]:
geo = gpd.read_file(RAW / 'geo_wards/London_Ward_CityMerged.shp')[['GSS_CODE', 'geometry']].rename(columns={'GSS_CODE': KEY})
cent = geo.geometry.centroid
data = data.merge(geo[[KEY]].assign(centroid_lon=cent.x, centroid_lat=cent.y), on=KEY)
assert data[KEY].is_unique and len(data) == 632, len(data)
data.to_csv(DATA / 'data_analysis_wards7s.csv', index=False)
gpd.GeoDataFrame(data.merge(geo, on=KEY), geometry='geometry', crs=geo.crs).to_file(DATA / 'data_analysis_wards7s_geo.geojson', driver='GeoJSON')
print(len(data), 'wards written;', list(data.columns))

632 wards written; ['ward_code', 'diabetes prescriptions', 'hypertension prescriptions', 'asthma prescriptions', 'opioids prescriptions', 'total prescriptions', 'SAT_NDVI', 'SAT_on_road_share_u', 'SAT_on_road_share_w', 'SAT_off_road', 'GSV_on_road_u', 'GSV_on_road_w', 'WHO', 'ESA_WHO_exposure', 'Natural_England1', 'median age', 'white percent', 'IMD deprivation score', 'building_density', 'centroid_lon', 'centroid_lat']


## Validation

In [6]:
data.describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
diabetes prescriptions,632.0,50.578,16.600,17.264,39.046,48.621,59.741,116.081
hypertension prescriptions,632.0,84.922,18.910,39.401,69.060,84.549,98.633,133.391
asthma prescriptions,632.0,14.127,2.505,6.602,12.269,13.987,15.862,22.751
opioids prescriptions,632.0,14.303,4.455,5.518,11.165,13.497,16.742,32.005
total prescriptions,632.0,1237.974,241.593,647.345,1090.952,1256.634,1389.613,2440.030
SAT_NDVI,632.0,0.414,0.154,0.057,0.301,0.407,0.500,0.913
SAT_on_road_share_u,632.0,0.225,0.078,0.043,0.171,0.214,0.265,0.717
SAT_on_road_share_w,632.0,0.203,0.071,0.042,0.155,0.193,0.240,0.533
SAT_off_road,632.0,0.074,0.089,0.000,0.018,0.047,0.091,0.604
GSV_on_road_u,632.0,0.074,0.021,0.024,0.059,0.073,0.088,0.149
